# Bronze File Ingestion - Olist Data



In [ ]:
# This notebook extracts the Brazilian Olist E-Commerce dataset from the 
# Kaggle website which includes 9 CSV files using the Kaggle REST API.

# Custom spark environment has been created for this notebook and the Kaggle
# python library added so that the Kaggle REST API can be used to support
# this task

# The Kaggle API Key has been added to the custom environment (env_Dev) used
# in this notebook

# created on: 12.09.2026
# Developer:  Asif Shah

In [ ]:
# This code checks whether the Kaggle python library is installed, otherwise installs it

import importlib
import subprocess
import sys

try: 
    kagglehub = importlib.import_module("kagglehub")
except ImportError: 
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
    kagglehub = importlib.import_module("kagglehub")

In [ ]:
# Pull the csv files from the Kaggle website
# API key is in the custom environment set up

# This code uses the Kaggle REST API library to extract the files from the Kaggle
# website and download them to the lakehouse files section

# Each run of this code will produce a new timestamp folder within the files section
# and new files within that folder. This supports enterprise grade ingestion patterns
# including metadata driven approaches

import os
import shutil
from datetime import datetime
import kagglehub

# Create a unique target path for each run using the current timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
target_path = f"Files/raw/{timestamp}"

# Download dataset to local temporary directory
download_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# Copy raw CSV files into the new timestamped folder
for file_name in os.listdir(download_path):
    if file_name.endswith('.csv'):
        local_file = os.path.join(download_path, file_name)
        mssparkutils.fs.cp(f"file:{local_file}", f"{target_path}/{file_name}", True)
        print(f"Loaded: {file_name} -> {target_path}/{file_name}")

# Clean up local temporary files
shutil.rmtree(download_path, ignore_errors=True)


In [ ]:
# The code in this cell updates the metadata_control table with the folder and file 
# names as well as the row counts of all the individual files within the timestamp 
# folder(s) not processed already

# This is important to later update and ensure incremental loads of files not processed
# in the lakehouse to the bronze tables
from pyspark.sql import functions as F

base_path = "Files/raw/*/*.csv"

# 1. Keep the underlying file content intact, but include metadata columns
spark.read.format("csv") \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("inferSchema", "true") \
    .load(base_path) \
    .selectExpr("*", "_metadata.file_path AS meta_path", "_metadata.file_name AS meta_name") \
    .createOrReplaceTempView("raw_csv_view")
  # .option("ignoreTrailingWhiteSpace", "true") \
  # .option("ignoreLeadingWhiteSpace", "true") \
  #  .option("escape", '"') \
  #  .option("quote", "\"")  \

spark.sql ("""

    INSERT INTO dbo.metadata_control

    SELECT
       b.meta_name,
       replace(replace(b.meta_name, "_dataset.csv", ""), ".csv", "")
      
    FROM raw_csv_view b
   
    WHERE NOT EXISTS
     (SELECT 1 
      FROM dbo.metadata_control a
      WHERE a.filename = b.meta_name)

    GROUP BY
    b.meta_name,
    replace(replace(b.meta_name, "_dataset.csv", ""), ".csv", "")

""")

# 2. Group by the metadata and count the actual underlying data records
spark.sql("""

    INSERT INTO dbo.audit_control
    SELECT 
        element_at(split(b.meta_path, '/'), -2) AS folder_name,
        b.meta_name AS file_name,
        COUNT(1) AS file_rows, -- This accurately counts the rows inside the CSVs
        a.tablename AS table_name,
        0,
        '',
        now()

    FROM raw_csv_view b
    JOIN dbo.metadata_control a 
      ON b.meta_name = a.filename

    WHERE NOT EXISTS (
        SELECT 1 
        FROM dbo.audit_control mc 
        WHERE mc.folder_name = element_at(split(b.meta_path, '/'), -2)
    )

    GROUP BY 
        element_at(split(b.meta_path, '/'), -2),
        b.meta_name,
        a.tablename

""")



In [4]:
# This cell truncates and reloads the unprocessed folders data after each CSV batch
# download to a new timestamped folder to ensure the lookup activity in the Olist
# Bronze Data Pipeline only refers to the folders where files have been processed

spark.sql("TRUNCATE TABLE dbo.unprocessed_folders")

spark.sql("""

    INSERT INTO dbo.unprocessed_folders
    SELECT
        folder_name,
        file_name,
        table_name

    FROM dbo.audit_control
    WHERE Status = '' or Status IS NULL

""")

StatementMeta(, 5afdabcc-8db0-422d-8402-c2f20654bdb7, 8, Finished, Available, Finished, False)

DataFrame[]